# 02 · ECG feature ablation

Run after `01_data_inspection.ipynb` creates the grouped split. This notebook extracts F1/F2 features and fits the 15 fixed model–feature configurations on that split. Set `APNEA_PHASE2_RUN` to a new directory for another run. Historical Phase 1 assignments remain archived for provenance.

In [ ]:
import hashlib
import json
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PHASE1 = Path(os.environ.get("APNEA_RUN", str(ROOT / "data/processed/phase1")))
PHASE2 = Path(os.environ.get("APNEA_PHASE2_RUN", str(ROOT / "data/processed/phase2-grouped")))
if not PHASE1.is_absolute():
    PHASE1 = ROOT / PHASE1
if not PHASE2.is_absolute():
    PHASE2 = ROOT / PHASE2
SPLIT_FILE = os.environ.get("APNEA_PHASE2_SPLITS")
SPLIT_RUN = Path(os.environ.get("APNEA_SPLIT_RUN", str(ROOT / "data/processed/phase1-grouped")))
SPLIT_FILE = Path(SPLIT_FILE) if SPLIT_FILE else SPLIT_RUN / "subject_splits.csv"
if SPLIT_FILE is not None and not SPLIT_FILE.is_absolute():
    SPLIT_FILE = ROOT / SPLIT_FILE
SEED = 67
print("Input:", PHASE1, "\nOutput:", PHASE2)

## Implementation · Feature extraction

The full RR feature calculations are defined in the next cell.

In [ ]:
FEATURES = ("mean_rr_ms", "median_rr_ms", "mean_hr_bpm", "sdrr_ms", "rmssd_ms", "prr50_pct")
SETS = {
    "A_F1": FEATURES[:3],
    "B_F2": FEATURES,
    "C_no_SDRR": tuple(x for x in FEATURES if x != "sdrr_ms"),
    "D_no_RMSSD": tuple(x for x in FEATURES if x != "rmssd_ms"),
    "E_no_pRR50": tuple(x for x in FEATURES if x != "prr50_pct"),
}


def extract_features(
    windows: pd.DataFrame, rr: pd.DataFrame, min_intervals: int = 20, min_pairs: int = 10
) -> pd.DataFrame:
    """Retain every window; mark undefined features NaN for train-only imputation.

    Only Phase 1 plausible intervals with both endpoints inside a window are
    accepted. A rejected interval breaks succession even if its neighbors pass.
    """
    if min_intervals < 2 or min_pairs < 1:
        raise ValueError("Minimum interval and pair counts must be positive")
    windows = windows.reset_index(drop=True)
    if windows.duplicated(["record_id", "minute"]).any():
        raise ValueError("Duplicate record/minute windows")
    columns = [*FEATURES, "n_feature_rr", "n_feature_pairs"]
    values = np.full((len(windows), len(columns)), np.nan)
    values[:, -2:] = 0
    for record, group in windows.groupby("record_id", sort=False):
        intervals = rr.loc[rr.record_id == record].sort_values("start_sample")
        starts = intervals.start_sample.to_numpy()
        ends = intervals.end_sample.to_numpy()
        seconds = intervals.rr_seconds.to_numpy(dtype=float)
        plausible = intervals.plausible.to_numpy(dtype=bool)
        for index, row in group.iterrows():
            lo = np.searchsorted(starts, row.start_sample, side="left")
            hi = np.searchsorted(starts, row.end_sample, side="left")
            inside = (ends[lo:hi] < row.end_sample) & (starts[lo:hi] >= row.start_sample)
            accepted = inside & plausible[lo:hi] & np.isfinite(seconds[lo:hi])
            accepted &= seconds[lo:hi] > 0
            selected = seconds[lo:hi][accepted] * 1000
            count = len(selected)
            adjacent = accepted[:-1] & accepted[1:]
            adjacent &= ends[lo:hi][:-1] == starts[lo:hi][1:]
            differences = np.diff(seconds[lo:hi] * 1000)[adjacent]
            pairs = len(differences)
            values[index, -2:] = count, pairs
            if count >= min_intervals:
                values[index, :4] = (
                    np.mean(selected),
                    np.median(selected),
                    np.mean(60000 / selected),
                    np.std(selected, ddof=1),
                )
            if count >= min_intervals and pairs >= min_pairs:
                values[index, 4:6] = (
                    np.sqrt(np.mean(differences**2)),
                    100 * np.mean(np.abs(differences) > 50),
                )
    result = windows.copy()
    result[columns] = values
    result["n_feature_rr"] = result.n_feature_rr.astype(int)
    result["n_feature_pairs"] = result.n_feature_pairs.astype(int)
    return result

## Implementation · Controlled model experiment

The next cell defines fixed classifiers, validation threshold selection, metrics, and artifact saving.

In [ ]:
def apply_split_policy(
    windows: pd.DataFrame, phase1_dir: Path, split_file: Path | None
) -> pd.DataFrame:
    """Keep Phase 1 files intact; optionally apply a verified sidecar group split."""
    if split_file is None:
        return windows
    sidecar = pd.read_csv(split_file)
    required = {"record_id", "subject_group", "split", "phase1_split"}
    if not required.issubset(sidecar) or sidecar.record_id.duplicated().any():
        raise ValueError("Invalid grouped split sidecar")
    original = pd.read_csv(phase1_dir / "splits.csv")
    check = sidecar[["record_id", "phase1_split"]].rename(columns={"phase1_split": "split"})
    if (
        not check.sort_values("record_id")
        .reset_index(drop=True)
        .equals(original[["record_id", "split"]].sort_values("record_id").reset_index(drop=True))
    ):
        raise ValueError("Grouped sidecar does not match the saved Phase 1 assignments")
    if sidecar.groupby("subject_group").split.nunique().max() != 1:
        raise ValueError("A subject group crosses the grouped split")
    result = (
        windows.drop(columns=["split", "group_id"])
        .merge(
            sidecar[["record_id", "subject_group", "split"]], on="record_id", validate="many_to_one"
        )
        .rename(columns={"subject_group": "group_id"})
    )
    if len(result) != len(windows):
        raise ValueError("Grouped sidecar omitted Phase 1 windows")
    return result


def validate_cohort(table: pd.DataFrame) -> None:
    if set(table.split) != {"train", "validation", "test"}:
        raise ValueError("Cohort must contain exactly train, validation, test partitions")
    if table.duplicated(["record_id", "minute"]).any():
        raise ValueError("Duplicate record/minute windows")
    if set(table.label) != {0, 1}:
        raise ValueError("Cohort must have binary labels")
    if table.groupby("group_id").split.nunique().max() != 1:
        raise ValueError("A source group crosses partitions")
    if table.groupby("record_id").split.nunique().max() != 1:
        raise ValueError("A recording crosses partitions")
    if any(set(group.label) != {0, 1} for _, group in table.groupby("split")):
        raise ValueError("Each partition must contain both classes")


def choose_threshold(labels: np.ndarray, probabilities: np.ndarray) -> float:
    """Maximize validation F1-score; select the lowest threshold on ties."""
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    scores = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(scores)])


def _model(name: str, seed: int):
    if name == "logistic_regression":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=1000, random_state=seed),
        )
    if name == "random_forest":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=200, max_depth=8, min_samples_leaf=5, n_jobs=4, random_state=seed
            ),
        )
    if name == "xgboost":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            XGBClassifier(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                subsample=1.0,
                colsample_bytree=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                n_jobs=4,
                random_state=seed,
            ),
        )
    raise ValueError(f"Unknown model: {name}")


def _metrics(labels, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "f1_score": f1_score(labels, predicted, zero_division=0),
        "pr_auc_average_precision": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "accuracy": accuracy_score(labels, predicted),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def run_ablation(
    phase1_dir: Path, output_dir: Path, seed: int = 67, split_file: Path | None = None
) -> pd.DataFrame:
    """Fit 15 fixed settings; select thresholds on validation, evaluate test once."""
    phase1_dir, output_dir = Path(phase1_dir), Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(f"Choose a new output directory; {output_dir} exists")
    manifest = json.loads((phase1_dir / "manifest.json").read_text())
    if manifest["split_status"] not in {"development_holdout", "unsplit_prepared"}:
        raise ValueError("Phase 2 requires the complete Phase 1 development holdout")
    windows = pd.read_parquet(phase1_dir / "windows.parquet")
    splits = pd.read_csv(phase1_dir / "splits.csv")
    if (
        not windows[["record_id", "group_id", "split"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .sort_values("record_id")
        .reset_index(drop=True)
        .equals(splits.sort_values("record_id").reset_index(drop=True))
    ):
        raise ValueError("Saved Phase 1 windows and split assignments disagree")
    windows = apply_split_policy(windows, phase1_dir, split_file)
    validate_cohort(windows)
    features = extract_features(windows, pd.read_parquet(phase1_dir / "rr.parquet"))
    if features.loc[features.split == "train", list(FEATURES)].isna().all().any():
        raise ValueError("A feature is entirely undefined in training")
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".phase2-", dir=output_dir.parent) as temporary:
        stage = Path(temporary) / "run"
        stage.mkdir()
        (stage / "models").mkdir()
        features.to_parquet(stage / "features.parquet", index=False)
        results, predictions = [], []
        for model_name in ("logistic_regression", "random_forest", "xgboost"):
            for set_name, columns in SETS.items():
                print(f"Fitting {model_name} / {set_name}", flush=True)
                train = features.split == "train"
                validation = features.split == "validation"
                classifier = _model(model_name, seed)
                classifier.fit(features.loc[train, list(columns)], features.loc[train, "label"])
                val_probability = classifier.predict_proba(features.loc[validation, list(columns)])[
                    :, 1
                ]
                threshold = choose_threshold(
                    features.loc[validation, "label"].to_numpy(), val_probability
                )
                for partition, mask, probability in (
                    ("validation", validation, val_probability),
                    ("test", features.split == "test", None),
                ):
                    if probability is None:
                        probability = classifier.predict_proba(features.loc[mask, list(columns)])[
                            :, 1
                        ]
                    labels = features.loc[mask, "label"].to_numpy()
                    results.append(
                        {
                            "model": model_name,
                            "feature_set": set_name,
                            "partition": partition,
                            "threshold": threshold,
                            "n_windows": len(labels),
                            **_metrics(labels, probability, threshold),
                        }
                    )
                    detail = features.loc[mask, ["record_id", "minute", "label", "split"]].copy()
                    detail["model"] = model_name
                    detail["feature_set"] = set_name
                    detail["probability"] = probability
                    detail["prediction"] = (probability >= threshold).astype(int)
                    predictions.append(detail)
                joblib.dump(
                    {"pipeline": classifier, "features": columns, "threshold": threshold},
                    stage / "models" / f"{model_name}_{set_name}.joblib",
                )
        metrics = pd.DataFrame(results)
        for metric in ("sensitivity", "specificity", "f1_score", "pr_auc_average_precision"):
            baseline = metrics[metrics.feature_set == "A_F1"].set_index(["model", "partition"])[
                metric
            ]
            full = metrics[metrics.feature_set == "B_F2"].set_index(["model", "partition"])[metric]
            key = pd.MultiIndex.from_frame(metrics[["model", "partition"]])
            metrics[f"delta_vs_A_F1_{metric}"] = (
                metrics[metric].to_numpy() - baseline.loc[key].to_numpy()
            )
            metrics[f"delta_vs_B_F2_{metric}"] = (
                metrics[metric].to_numpy() - full.loc[key].to_numpy()
            )
        metrics.to_csv(stage / "metrics.csv", index=False)
        pd.concat(predictions, ignore_index=True).to_parquet(
            stage / "predictions.parquet", index=False
        )
        (stage / "manifest.json").write_text(
            json.dumps(
                {
                    "created_utc": datetime.now(UTC).isoformat(),
                    "phase1_path": str(phase1_dir.resolve()),
                    "phase1_manifest_sha256": hashlib.sha256(
                        (phase1_dir / "manifest.json").read_bytes()
                    ).hexdigest(),
                    "seed": seed,
                    "n_windows": len(features),
                    "quality_policy": (
                        "retain all Phase 1 windows; reject implausible RR; no RR repair"
                    ),
                    "minimum_accepted_rr": 20,
                    "minimum_adjacent_pairs": 10,
                    "missing_policy": "undefined as NaN; median imputation fitted on training only",
                    "threshold_rule": "maximize validation F1-score; lowest threshold on ties",
                    "pr_auc_definition": "scikit-learn average_precision_score",
                    "independence": (
                        "literature-derived subject-group-disjoint; "
                        "original patient IDs unavailable"
                        if split_file is not None
                        else manifest["independence"]
                    ),
                    "split_policy": "grouped" if split_file is not None else "phase1_recording",
                    "split_file_sha256": hashlib.sha256(Path(split_file).read_bytes()).hexdigest()
                    if split_file is not None
                    else None,
                    "models": {
                        name: str(_model(name, seed))
                        for name in ("logistic_regression", "random_forest", "xgboost")
                    },
                    "feature_sets": {name: list(columns) for name, columns in SETS.items()},
                },
                indent=2,
            )
            + "\n"
        )
        stage.rename(output_dir)
    return metrics

## 1. Check the cohort and grouped split

Notebook 01 saves the grouped split after inspection. This notebook checks its lineage and group isolation before feature extraction. Original patient IDs remain unavailable.

In [ ]:
source = json.loads((PHASE1 / "manifest.json").read_text())
assert source["split_status"] in {"development_holdout", "unsplit_prepared"}
windows = apply_split_policy(pd.read_parquet(PHASE1 / "windows.parquet"), PHASE1, SPLIT_FILE)
validate_cohort(windows)
display(windows.groupby(["split", "label"]).size().unstack(fill_value=0))
print("Grouped split: literature-derived subject groups; patient IDs unverified")

## 2. Extract six RR features

For each minute, accept only Phase 1 plausible RR intervals whose **two beat endpoints** are inside that minute. Removing an invalid interval must not join its neighbors into a new pair. Require at least 20 accepted intervals for mean RR, median RR, mean instantaneous HR, and sample SDRR; require at least 10 valid originally adjacent pairs for RR-based RMSSD and pRR50. Undefined features remain `NaN`.

| Feature | Calculation |
| --- | --- |
| Mean RR | `mean(r)` ms |
| Median RR | `median(r)` ms |
| Mean HR | `mean(60000/r)` bpm |
| SDRR | `std(r, ddof=1)` ms |
| RR-based RMSSD | `sqrt(mean(d²))` ms |
| pRR50 | `100 × mean(abs(d) > 50)` %; denominator is the number of valid pairs |

Detected RR intervals have **not** been validated as normal-to-normal intervals.

In [ ]:
rr = pd.read_parquet(PHASE1 / "rr.parquet")
features = extract_features(windows, rr)
assert len(features) == len(windows)
display(
    pd.DataFrame({"feature": FEATURES, "missing": features[list(FEATURES)].isna().sum().to_numpy()})
)
display(features.groupby("split")[list(FEATURES)].median().round(2))
print("Feature configurations:")
for name, columns in SETS.items():
    print(name, "→", ", ".join(columns))

## 3. Fit the controlled experiment

Logistic Regression, Random Forest, and XGBoost are **retrained for each feature set** with identical model settings and random seed. Every configuration retains the same windows. Median imputation (and Logistic Regression scaling) is fitted on training data only. Each threshold maximizes validation F1-score, with the lowest threshold on ties; it is frozen before test evaluation. Precision–recall AUC means average precision.

If `PHASE2` exists, we read it only when its split provenance matches the selected policy. To rerun, set `APNEA_PHASE2_RUN` to a new directory before executing this notebook.

In [ ]:
if PHASE2.exists():
    print("Reusing experiment:", PHASE2)
else:
    run_ablation(PHASE1, PHASE2, seed=SEED, split_file=SPLIT_FILE)
    print("Saved experiment:", PHASE2)
metrics = pd.read_csv(PHASE2 / "metrics.csv")
predictions = pd.read_parquet(PHASE2 / "predictions.parquet")
manifest = json.loads((PHASE2 / "manifest.json").read_text())
assert len(metrics) == 30
assert set(metrics.feature_set) == set(SETS)
assert manifest["n_windows"] == len(features)
assert manifest.get("split_policy", "phase1_recording") == (
    "grouped" if SPLIT_FILE is not None else "phase1_recording"
)
if SPLIT_FILE is not None:
    assert manifest["split_file_sha256"] == hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()
print("Pipelines:", len(list((PHASE2 / "models").glob("*.joblib"))))
print("PR-AUC:", manifest["pr_auc_definition"])
print("Threshold:", manifest["threshold_rule"])

## 4. Compare validation and held-out test performance

The table includes sensitivity, specificity, F1-score, average precision, ROC-AUC, and accuracy. The test set is for reporting after all settings and thresholds are frozen; do not retune based on its results.

In [ ]:
columns = [
    "model",
    "feature_set",
    "partition",
    "threshold",
    "sensitivity",
    "specificity",
    "f1_score",
    "pr_auc_average_precision",
    "roc_auc",
    "accuracy",
]
display(metrics[columns].round(3))
test = metrics[metrics.partition == "test"]
display(
    test.pivot(index="model", columns="feature_set", values="pr_auc_average_precision").round(3)
)

## 5. Visualize the primary B versus A comparison

Average precision is threshold-free. F1-score depends on the chosen threshold; differences between validation and test can expose recording-level shift. Removal sets C, D, and E are conditional comparisons and do not establish causal feature importance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, title in zip(
    axes, ["pr_auc_average_precision", "f1_score"], ["Average precision", "F1-score"], strict=True
):
    table = test[test.feature_set.isin(["A_F1", "B_F2"])].pivot(
        index="model", columns="feature_set", values=metric
    )
    table.plot.bar(ax=ax, rot=15)
    ax.set(title=title, ylim=(0, 1), xlabel="Classifier")
display(fig)
plt.close(fig)
display(
    test[
        [
            "model",
            "feature_set",
            "delta_vs_A_F1_pr_auc_average_precision",
            "delta_vs_B_F2_pr_auc_average_precision",
            "delta_vs_A_F1_f1_score",
            "delta_vs_B_F2_f1_score",
        ]
    ].round(3)
)

## 6. Examine errors without changing the experiment

Select a model and feature set to summarize false positives and false negatives by recording. These are retrospective error counts, not grounds to update the saved threshold using the held-out test set.

In [ ]:
MODEL, FEATURE_SET = "logistic_regression", "B_F2"
errors = predictions[
    (predictions.model == MODEL)
    & (predictions.feature_set == FEATURE_SET)
    & (predictions.split == "test")
].copy()
errors["outcome"] = np.select(
    [
        errors.label.eq(1) & errors.prediction.eq(1),
        errors.label.eq(0) & errors.prediction.eq(0),
        errors.label.eq(0) & errors.prediction.eq(1),
    ],
    ["TP", "TN", "FP"],
    default="FN",
)
display(errors.groupby(["record_id", "outcome"]).size().unstack(fill_value=0))

## Interpretation and limits

See [Phase 2 results](../docs/phase-2-results.md) for the written summary. These one-minute detected-beat features do not establish conventional long-duration HRV or a specific physiological mechanism. Patient identity resolution, beat audit, and independent validation are still needed. This is a research screening prototype, not a clinical diagnostic system.